# Microcosmic God: Colab A100 Run

This notebook runs a large sealed Microcosmic God experiment on Google Colab and bundles the outputs for later analysis.

The current simulator is Python/CPU-bound, so the world step itself still runs on CPU. This notebook uses the A100 for tensor-shaped post-run checkpoint analysis through the CUDA/PyTorch SAE backend.

Captured outputs include `config.json`, `events.jsonl`, `story_events.jsonl`, `summary.json`, `world_final.json`, checkpoints, text reports, derived CSV/JSON indexes, a manifest, and a zip archive.

## 1. Run Settings

Defaults are tuned for a 10-minute story-rich run. `FULL_EVENT_DETAIL = False` keeps routine birth/death/tool-detail spam out of `events.jsonl` while preserving aggregate telemetry, story events, final world state, and checkpoints. Flip it to `True` only if you intentionally want much larger raw event logs.

In [ ]:
REPO_URL = "https://github.com/asystemoffields/microcosmic-god.git"
REPO_REF = "codex/colab-a100-run-notebook"

RUN_NAME = "a100_10m_seed1958"
SEED = 1958

# Modal-sized world, bounded by wall time.
PROFILE = "modal"
WALL_SECONDS = 10 * 60
MAX_TICKS = 10_000_000
PLACES = 256
PLANTS = 4_000
FUNGI = 1_000
AGENTS = 800
MAX_POPULATION = 80_000

# Frequent enough for a good story timeline, sparse enough for long-run safety.
LOG_EVERY = 1_000
CHECKPOINT_EVERY = 5_000
CHECKPOINT_LIMIT = 256
FULL_EVENT_DETAIL = False

# Post-run analysis.
TRAIN_SAE = True
SAE_LATENT = 32
SAE_STEPS = 1_500
SAE_BACKEND = "torch"
SAE_DEVICE = "cuda"

# Save a copy to Google Drive when running in Colab.
MOUNT_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/microcosmic_god_runs"

LOCAL_ROOT = "/content/microcosmic_god_colab"
REPO_DIR = "/content/microcosmic-god"

## 2. Environment Check

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import shlex
import shutil
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path


def run(cmd, cwd=None, check=True, capture=False):
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"$ {printable}")
    return subprocess.run(
        [str(part) for part in cmd],
        cwd=cwd,
        check=check,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )


print("Python", sys.version)
run(["nvidia-smi"], check=False)
try:
    import torch

    print("PyTorch", torch.__version__)
    print("torch.cuda.is_available", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device", torch.cuda.get_device_name(0))
except Exception as exc:
    print(f"PyTorch CUDA check skipped: {exc}")
run([sys.executable, "-m", "pip", "--version"], check=True)
Path(LOCAL_ROOT).mkdir(parents=True, exist_ok=True)

## 3. Optional Drive Mount

In [ ]:
drive_dir = None
if MOUNT_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        drive_dir = Path(DRIVE_OUTPUT_DIR)
        drive_dir.mkdir(parents=True, exist_ok=True)
        print(f"Drive output: {drive_dir}")
    except Exception as exc:
        print(f"Drive mount skipped or failed: {exc}")
        drive_dir = None

## 4. Clone And Install

In [ ]:
repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    run(["git", "-C", repo_dir, "fetch", "--all", "--tags"])
else:
    run(["git", "clone", REPO_URL, repo_dir])

run(["git", "checkout", REPO_REF], cwd=repo_dir)
run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=repo_dir, check=False)
run([sys.executable, "-m", "pip", "install", "-e", ".[analysis]"], cwd=repo_dir)
run([sys.executable, "-m", "microcosmic_god", "specs"], cwd=repo_dir)

## 5. Dry Run Card

Read this card before launching the real run. It should say `mode: sealed`, a 600 second wall limit, and the requested population/world sizes.

In [ ]:
run_root = Path(LOCAL_ROOT) / RUN_NAME
runs_dir = run_root / "runs"
reports_dir = run_root / "reports"
runs_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

base_cmd = [
    sys.executable,
    "-m",
    "microcosmic_god",
    "run",
    "--profile",
    PROFILE,
    "--seed",
    SEED,
    "--ticks",
    MAX_TICKS,
    "--wall-seconds",
    WALL_SECONDS,
    "--places",
    PLACES,
    "--plants",
    PLANTS,
    "--fungi",
    FUNGI,
    "--agents",
    AGENTS,
    "--max-population",
    MAX_POPULATION,
    "--output-dir",
    runs_dir,
    "--log-every",
    LOG_EVERY,
    "--checkpoint-every",
    CHECKPOINT_EVERY,
    "--checkpoint-limit",
    CHECKPOINT_LIMIT,
]
if not FULL_EVENT_DETAIL:
    base_cmd.append("--quiet-events")

run([*base_cmd, "--dry-run"], cwd=repo_dir)

## 6. Launch Run

This cell streams output and writes a durable stdout/stderr log next to the run artifacts.

In [ ]:
stdout_log = run_root / f"{RUN_NAME}_stdout_stderr.log"
existing_dirs = {path.resolve() for path in runs_dir.glob("*") if path.is_dir()}
started_at = datetime.now(timezone.utc).isoformat()
print(f"Started at UTC: {started_at}")

with stdout_log.open("w", encoding="utf-8") as handle:
    process = subprocess.Popen(
        [str(part) for part in base_cmd],
        cwd=repo_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        handle.write(line)
    return_code = process.wait()

finished_at = datetime.now(timezone.utc).isoformat()
print(f"Finished at UTC: {finished_at}")
if return_code != 0:
    raise RuntimeError(f"simulation exited with code {return_code}; see {stdout_log}")

new_dirs = [path for path in runs_dir.glob("*") if path.is_dir() and path.resolve() not in existing_dirs]
if not new_dirs:
    new_dirs = [path for path in runs_dir.glob("*") if path.is_dir()]
run_dir = max(new_dirs, key=lambda path: path.stat().st_mtime)
print(f"Run directory: {run_dir}")

## 7. Validate Required Artifacts

In [ ]:
required_files = ["config.json", "events.jsonl", "story_events.jsonl", "summary.json", "world_final.json"]
missing = [name for name in required_files if not (run_dir / name).exists()]
if missing:
    raise FileNotFoundError(f"missing run artifacts: {missing}")
if not (run_dir / "checkpoints").is_dir():
    raise FileNotFoundError("missing checkpoints directory")


def line_count(path: Path) -> int:
    with path.open("r", encoding="utf-8") as handle:
        return sum(1 for _ in handle)


summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
print(json.dumps({
    "reason": summary.get("reason"),
    "tick": summary.get("tick"),
    "elapsed_seconds": summary.get("elapsed_seconds"),
    "population": summary.get("population"),
    "births_by_mode": summary.get("births_by_mode"),
    "deaths_by_cause": summary.get("deaths_by_cause"),
    "tool_successes": summary.get("tool_successes"),
    "checkpointing": summary.get("checkpointing"),
}, indent=2, sort_keys=True))

print("events.jsonl lines:", line_count(run_dir / "events.jsonl"))
print("story_events.jsonl lines:", line_count(run_dir / "story_events.jsonl"))
print("checkpoint files:", len(list((run_dir / "checkpoints").glob("brain_*.json"))))

## 8. Generate Reports And Indexes

In [ ]:
def run_to_file(cmd, out_path: Path, cwd=None):
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"$ {printable} > {out_path}")
    completed = subprocess.run(
        [str(part) for part in cmd],
        cwd=cwd,
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    out_path.write_text(completed.stdout, encoding="utf-8")
    print(completed.stdout[:4000])


run_to_file([sys.executable, "analysis/scripts/summarize_run.py", run_dir], reports_dir / "summary_report.txt", cwd=repo_dir)
run_to_file([sys.executable, "analysis/scripts/story_report.py", run_dir], reports_dir / "story_report.txt", cwd=repo_dir)


def read_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


events = read_jsonl(run_dir / "events.jsonl")
stories = read_jsonl(run_dir / "story_events.jsonl")
aggregates = [row for row in events if row.get("kind") == "aggregate"]

story_counts = Counter(row.get("kind", "unknown") for row in stories)
(reports_dir / "story_kind_counts.json").write_text(json.dumps(story_counts, indent=2, sort_keys=True) + "\n", encoding="utf-8")

top_stories = sorted(stories, key=lambda row: row.get("interestingness", 0.0), reverse=True)[:250]
(reports_dir / "top_story_events.json").write_text(json.dumps(top_stories, indent=2, sort_keys=True) + "\n", encoding="utf-8")

timeline_path = reports_dir / "aggregate_timeline.csv"
with timeline_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=[
        "tick",
        "total_population",
        "neural_population",
        "agent_population",
        "births_total",
        "deaths_total",
        "tool_success_total",
        "marks_total",
        "structures_total",
        "avg_move_energy_cost",
        "avg_relocation_shock",
    ])
    writer.writeheader()
    for row in aggregates:
        pop = row.get("population", {})
        movement = row.get("movement", {})
        writer.writerow({
            "tick": row.get("tick", 0),
            "total_population": pop.get("total", 0),
            "neural_population": pop.get("neural", 0),
            "agent_population": pop.get("agent", 0),
            "births_total": sum(row.get("births", {}).values()),
            "deaths_total": sum(row.get("deaths", {}).values()),
            "tool_success_total": sum(row.get("tool_successes", {}).values()),
            "marks_total": sum(row.get("marks_created", {}).values()),
            "structures_total": sum(row.get("structures_built", {}).values()) + sum(row.get("structures_extended", {}).values()),
            "avg_move_energy_cost": movement.get("avg_energy_cost", 0.0),
            "avg_relocation_shock": movement.get("avg_relocation_shock", 0.0),
        })

checkpoint_index_path = reports_dir / "checkpoint_index.csv"
with checkpoint_index_path.open("w", newline="", encoding="utf-8") as handle:
    fieldnames = ["file", "tick", "reason", "organism_id", "age", "generation", "energy", "offspring", "successful_tools", "complexity", "last_action", "last_tool_affordance"]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for checkpoint in sorted((run_dir / "checkpoints").glob("brain_*.json")):
        data = json.loads(checkpoint.read_text(encoding="utf-8"))
        organism = data.get("organism", {})
        writer.writerow({
            "file": checkpoint.name,
            "tick": data.get("tick", ""),
            "reason": data.get("reason", ""),
            "organism_id": organism.get("id", ""),
            "age": organism.get("age", ""),
            "generation": organism.get("generation", ""),
            "energy": organism.get("energy", ""),
            "offspring": organism.get("offspring_count", ""),
            "successful_tools": organism.get("successful_tools", ""),
            "complexity": organism.get("complexity", ""),
            "last_action": organism.get("last_action", ""),
            "last_tool_affordance": organism.get("last_tool_affordance", ""),
        })

print("Wrote reports to", reports_dir)

## 9. Optional SAE On Checkpoints

In [ ]:
if TRAIN_SAE:
    checkpoint_count = len(list((run_dir / "checkpoints").glob("brain_*.json")))
    if checkpoint_count == 0:
        print("Skipping SAE: no checkpoints.")
    else:
        sae_out = reports_dir / "checkpoint_sae.npz"
        run([
            sys.executable,
            "analysis/scripts/train_checkpoint_sae.py",
            run_dir / "checkpoints",
            "--latent",
            SAE_LATENT,
            "--steps",
            SAE_STEPS,
            "--backend",
            SAE_BACKEND,
            "--device",
            SAE_DEVICE,
            "--out",
            sae_out,
        ], cwd=repo_dir)
else:
    print("Skipping SAE by configuration.")

## 10. Manifest And Zip Bundle

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "run_name": RUN_NAME,
    "run_dir": str(run_dir),
    "settings": {
        "profile": PROFILE,
        "seed": SEED,
        "wall_seconds": WALL_SECONDS,
        "max_ticks": MAX_TICKS,
        "places": PLACES,
        "plants": PLANTS,
        "fungi": FUNGI,
        "agents": AGENTS,
        "max_population": MAX_POPULATION,
        "log_every": LOG_EVERY,
        "checkpoint_every": CHECKPOINT_EVERY,
        "checkpoint_limit": CHECKPOINT_LIMIT,
        "full_event_detail": FULL_EVENT_DETAIL,
        "sae_backend": SAE_BACKEND,
        "sae_device": SAE_DEVICE,
    },
    "summary": summary,
    "files": [],
}

for root in [run_dir, reports_dir]:
    for path in sorted(root.rglob("*")):
        if path.is_file():
            manifest["files"].append({
                "path": str(path.relative_to(run_root)),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            })

manifest_path = run_root / "MANIFEST.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")

archive_base = Path(LOCAL_ROOT) / f"{RUN_NAME}_{run_dir.name}_bundle"
zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=run_root)
print(f"Bundle: {zip_path}")
print(f"Bundle bytes: {Path(zip_path).stat().st_size:,}")

if drive_dir is not None:
    destination = drive_dir / Path(zip_path).name
    shutil.copy2(zip_path, destination)
    shutil.copy2(manifest_path, drive_dir / f"{RUN_NAME}_{run_dir.name}_MANIFEST.json")
    print(f"Copied bundle to: {destination}")
    print(f"Copied manifest to: {drive_dir}")

## 11. Quick Artifact Preview

In [ ]:
print("Run dir:", run_dir)
print("Reports dir:", reports_dir)
print("Manifest:", manifest_path)
print("Zip:", zip_path)
print("Story counts:")
print((reports_dir / "story_kind_counts.json").read_text(encoding="utf-8"))
print("Summary report preview:")
print((reports_dir / "summary_report.txt").read_text(encoding="utf-8")[:4000])